# Feature B — Model 2: Gemini Flash-Lite

This notebook isolates and tests the secondary sentiment model. It checks which Flash-Lite model the API key can access, enforces structured batch output, evaluates manually checked labels, and optionally cross-checks labels previously created by Model 1.

In [ ]:
!pip install -q -U google-genai python-dotenv pandas pyarrow tqdm scikit-learn

In [ ]:
import os, sys
from pathlib import Path
from getpass import getpass
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

from google.colab import drive
drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/Amazon Mini Projects/Amazon-Marketplace-Product-Intelligence')
assert PROJECT.exists(), f'Incorrect project path: {PROJECT}'
sys.path.insert(0, str(PROJECT))

if not os.getenv('GEMINI_API_KEY'):
    os.environ['GEMINI_API_KEY'] = getpass('Gemini API key: ')

from analysis.labelling_model2 import (
    GeminiLabeller, create_gold_set, resolve_model_name, VALID_SENTIMENTS
)

REQUEST_DELAY = 6.5
MODEL_1 = 'gemini-3.6-flash'
MODEL_2 = resolve_model_name(
    preferred='gemini-3.5-flash-lite',
    fallbacks=('gemini-3.1-flash-lite', 'gemini-2.5-flash-lite'),
)
print('Model 2 selected:', MODEL_2)

## 1. Model 2 pre-flight test

Do not continue to hundreds of reviews unless this single request succeeds. A 404 means the model is unavailable to the key; a 429 means quota/rate limiting; 401/403 indicates key or project permissions.

In [ ]:
model2 = GeminiLabeller(model_name=MODEL_2, batch_size=10, request_delay=REQUEST_DELAY)
if not model2.test_connection():
    raise RuntimeError('Model 2 connection failed. Read the API code printed above before continuing.')

## 2. Manually checked Model 2 evaluation

Replace or extend these examples with real reviews you personally checked. API failures are excluded from accuracy and reported separately.

In [ ]:
manual_test_set = [
    {'text': 'I really wanted to hate this, but it actually works perfectly.', 'ground_truth': 'positive'},
    {'text': 'Not bad for the price, but it broke after a week.', 'ground_truth': 'negative'},
    {'text': "It's exactly a standard mouse. Nothing special.", 'ground_truth': 'neutral'},
    {'text': 'Oh great, another charger that stopped working in two days.', 'ground_truth': 'negative'},
    {'text': "Does the job. Wouldn't buy again but wouldn't call it bad either.", 'ground_truth': 'neutral'},
    {'text': 'Shipping was slow but the product itself is fantastic.', 'ground_truth': 'positive'},
    {'text': 'Five stars because the seller was nice. Product is mediocre.', 'ground_truth': 'neutral'},
    {'text': 'Bought it for my kid; she loves it, although the battery dies fast.', 'ground_truth': 'positive'},
    {'text': 'Returned it. Enough said.', 'ground_truth': 'negative'},
    {'text': 'Works as described. Nothing more, nothing less.', 'ground_truth': 'neutral'},
]

texts = [row['text'] for row in manual_test_set]
truth = [row['ground_truth'] for row in manual_test_set]
prediction = model2.label_batch(texts)
results = pd.DataFrame({'review': texts, 'ground_truth': truth, 'model2_prediction': prediction})
results['api_success'] = results['model2_prediction'].isin(VALID_SENTIMENTS)
results['correct'] = results['ground_truth'].eq(results['model2_prediction']) & results['api_success']
display(results)

valid = results[results['api_success']]
print('Successful predictions:', len(valid), '/', len(results))
print('API/parse failures:', len(results) - len(valid))
if valid.empty:
    raise RuntimeError('Model 2 produced no usable predictions.')
print('Accuracy:', round(accuracy_score(valid['ground_truth'], valid['model2_prediction']), 4))
print('Macro-F1:', round(f1_score(valid['ground_truth'], valid['model2_prediction'], average='macro'), 4))
print(classification_report(valid['ground_truth'], valid['model2_prediction'], labels=['positive','neutral','negative'], zero_division=0))

## 3. Optional cross-check against Model 1 labels

Run this only after your review DataFrame already contains `llm_sentiment` from Model 1. Agreement is diagnostic—not manual ground truth. Do not report Model 1 labels as human gold labels.

In [ ]:
# Example after loading your cleaned reviews:
# from analysis.cleaning import load_clean
# products, reviews = load_clean()
# labelled_reviews = reviews[reviews['llm_sentiment'].isin(VALID_SENTIMENTS)].copy()
# gold = create_gold_set(
#     labelled_reviews, model1=MODEL_1, model2=MODEL_2,
#     n_agree=120, n_disagree=30, save=True,
# )
# print('Inter-model agreement:', gold['agreement'].mean())

print('Model 2 is ready. Uncomment this section only when Model 1 labels exist.')